# Actividad 8 – Clasificación de fraude en pagos (MLlib)
### Iván López - A01284875

## Metodología
* Leer el archivo csv con PySpark.
* Preparar la etiqueta y features
* Separar datos en train y test
* Pipeline de ML (preprocesamiento y LogisticRegression)
* Evaluar el modelo
* Discusión y reflexión

## Librerías requeridas

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

## Configuración de Spark

In [2]:
# Crear sesión Spark
try:
    sc.stop()
except Exception:
    pass

In [3]:
spark = (SparkSession.builder
         .appName("NotebookSession")
         .master("local[*]")
         .config("spark.ui.port", "0")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(spark.version, sc.appName)

4.0.1 NotebookSession


## Carga de datos

In [15]:
# leer el CSV
df_pay = spark.read.csv('activity8_payments_big.csv', header=True, inferSchema=True) 

In [16]:
df_pay = (df_pay
          .withColumn('tx_ts', F.to_timestamp('tx_ts'))
          .withColumn('amount', F.col('amount').cast('double')))

In [17]:
# verificar esquema
df_pay.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_fraud_label: integer (nullable = true)



In [18]:
#primeras filas
df_pay.show(10)

+--------------------+-----------+-----------+-------+--------+-------------------+--------+-------+-------+--------------+
|               tx_id|customer_id|merchant_id| amount|currency|              tx_ts|  status|channel|country|is_fraud_label|
+--------------------+-----------+-----------+-------+--------+-------------------+--------+-------+-------+--------------+
|0d3ad28b-bcbf-4a4...|      55377|       1276| 541.12|     MXN|2025-05-08 09:16:00|DECLINED|    APP|     MX|             0|
|c20fb64f-42b6-4f3...|      33277|       4530|3631.56|     MXN|2025-04-13 20:13:00|APPROVED|    APP|     US|             0|
|caba3ec1-6010-417...|       1359|       2157| 2388.0|     MXN|2025-04-07 13:03:00|APPROVED|    CNP|     BR|             0|
|a1503cbe-d809-43e...|      16633|       2220| 654.48|     MXN|2025-04-04 23:18:00|DECLINED|    CNP|     AR|             0|
|b33d5ab4-15c9-46a...|      72886|        493| 874.95|     MXN|2025-04-06 18:00:00|APPROVED|PRESENT|     CO|             0|
|a9a2e06

## Preparar la etiqueta y features

In [19]:
#columna label, high_value 
df_pay=(df_pay
        .withColumn('label', F.col('is_fraud_label').cast('int'))
        .withColumn('is_high_value', (F.when(F.col('amount') > 2000, 1).otherwise(0))))

In [20]:
#convertir las categorías a índices numéricos
idxs = StringIndexer(
    inputCols=['channel', 'country'], 
    outputCols=['channel_idx', 'country_idx']
)

#fit y transform a las columnas
df_pay = idxs.fit(df_pay).transform(df_pay)

In [22]:
df_pay=(df_pay
        .withColumn('channel_idx', F.col('channel_idx').cast('int'))
        .withColumn('country_idx', F.col('country_idx').cast('int')))

df_pay.show(10)

+--------------------+-----------+-----------+-------+--------+-------------------+--------+-------+-------+--------------+-----+-------------+-----------+-----------+
|               tx_id|customer_id|merchant_id| amount|currency|              tx_ts|  status|channel|country|is_fraud_label|label|is_high_value|channel_idx|country_idx|
+--------------------+-----------+-----------+-------+--------+-------------------+--------+-------+-------+--------------+-----+-------------+-----------+-----------+
|0d3ad28b-bcbf-4a4...|      55377|       1276| 541.12|     MXN|2025-05-08 09:16:00|DECLINED|    APP|     MX|             0|    0|            0|          2|          4|
|c20fb64f-42b6-4f3...|      33277|       4530|3631.56|     MXN|2025-04-13 20:13:00|APPROVED|    APP|     US|             0|    0|            1|          2|          0|
|caba3ec1-6010-417...|       1359|       2157| 2388.0|     MXN|2025-04-07 13:03:00|APPROVED|    CNP|     BR|             0|    0|            1|          1|     

## Separación de datos en train/test

In [23]:
# Dividimos 70% para train y 30% para test
train, test = df_pay.randomSplit([0.7, 0.3], seed=42)

## Pipeline de ML

In [25]:
#Combinar columnas en un solo vector
assembler = VectorAssembler(
    inputCols=['amount', 'is_high_value', 'channel_idx', 'country_idx'],
    outputCol='raw_features'
)

#Escalar los datos 
scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withStd=True,
    withMean=False 
)

#modelo
lr = LogisticRegression(
    featuresCol='features', 
    labelCol='label'
)

In [26]:
#Pipeline con las etapas en orden
pipeline = Pipeline(stages=[assembler, scaler, lr])

In [27]:
# entrenamiento
model = pipeline.fit(train)

## Evaluación del modelo

In [29]:
pred = model.transform(test) #predicciones

In [31]:
#AUC
eval = BinaryClassificationEvaluator(
    rawPredictionCol='rawPrediction', 
    labelCol='label',
    metricName='areaUnderROC'
)

auc = eval.evaluate(pred)
print(f'AUC: {auc:.4f}')

AUC: 0.5085


## Resultados de la predicciones

In [37]:
pred_results=(pred
              .select('tx_id', 'amount', 'label', 'probability', 'prediction')
              .withColumn('prediction', F.col('prediction').cast('int')))

pred_results.show(10, truncate=False)

+------------------------------------+-------+-----+-----------------------------------------+----------+
|tx_id                               |amount |label|probability                              |prediction|
+------------------------------------+-------+-----+-----------------------------------------+----------+
|00033f2f-f7d1-4f04-80f2-ccd889a6e6b0|2354.54|0    |[0.9633014464849572,0.03669855351504281] |0         |
|0009cf51-5e3c-458d-b5a9-a35759a54f80|2765.97|0    |[0.958229340755866,0.041770659244134034] |0         |
|000f2c22-1198-4981-abd1-d0fe54518539|2334.61|0    |[0.9576971783197363,0.04230282168026367] |0         |
|00142d92-f278-4eeb-b0ed-0a201f3e1f21|2310.95|0    |[0.9589727460060774,0.0410272539939226]  |0         |
|001c507e-dc6c-43ce-81c4-81e2cf84987c|1917.21|0    |[0.9573098339947564,0.04269016600524356] |0         |
|001d1120-8adf-4c0a-b070-3c5ac6515991|4271.26|0    |[0.9622445750425186,0.03775542495748141] |0         |
|0021ed02-7023-4e92-b14a-f9b040adfe17|2801.54|

## Discusión y reflexión
La AUC obtenida de 0.5085 no es buena, pues indica que el modelo apenas supera el azar en 0.5 y no tiene capacidad real para distinguir clases, lo cual lo hace no tan útil para la toma de decisiones a como está actualmente. Además, al observar la tabla de predicciones, se ve en los primeros registros que las probabilidades de fraude son muy bajas (alrededor de 0.03-0.04) y la predicción es siempre 0, lo que revela la limitación principal de este enfoque simplificado que usa variables estáticas como amount, country y channel que en realidad no describen patrones delictivos complejos. Para mejorarlo, se deben agregar features de comportamiento como velocidad de las transacciones, tiempo desde el último pago, desviación del monto promedio histórico, etc. que permitan al modelo detectar anomalías auténticas, tangibles y sospechosas más allá de un monto alto o un país específico.